# Encore – Fase 0: Validação de Dados

Notebook exploratório para validar a qualidade e a cobertura dos dados das APIs
**setlist.fm** (setlists) e **MusicBrainz** (discografia) antes de construir qualquer pipeline.

Bandas analisadas: Arctic Monkeys, Oasis, Linkin Park, Twenty One Pilots, Muse, Metallica, Avenged Sevenfold.

## Configuração Global

In [ ]:
# ============================================================
# Parâmetro principal: quantas páginas de setlists buscar
# por artista. Comece com 2 para testar; use 999 para carga completa.
MAX_PAGES = 999
# ============================================================

import sys
import re
import unicodedata
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Exibe tabelas completas sem truncamento
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

# Adiciona a raiz do projeto ao path para importar src.clients
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.clients import (
    SetlistFmClient,
    MusicBrainzClient,
    get_counters,
    get_cache_hits,
    reset_counters,
)

# Instancia os clientes uma única vez
slclient = SetlistFmClient()
mbclient = MusicBrainzClient()

# Lista de bandas para análise
BANDAS = [
    "Arctic Monkeys",
    "Oasis",
    "Linkin Park",
    "Twenty One Pilots",
    "Muse",
    "Metallica",
    "Avenged Sevenfold",
]

# ---------------------------------------------------------------
# Contadores acumulados da sessão (não zeram entre células)
# ---------------------------------------------------------------
# Usamos get_counters() / get_cache_hits() para os deltas por etapa,
# mas mantemos totais acumulados manualmente para a etapa 9.
_total_req  = {"setlistfm": 0, "musicbrainz": 0}
_total_cache = {"setlistfm": 0, "musicbrainz": 0}

def _atualiza_totais():
    """Soma os contadores correntes nos totais acumulados da sessão."""
    c = get_counters()
    h = get_cache_hits()
    for api in ("setlistfm", "musicbrainz"):
        _total_req[api]   += c[api]
        _total_cache[api] += h[api]

def _print_contadores(label=""):
    c = get_counters()
    h = get_cache_hits()
    if label:
        print(f"\n[{label}]")
    print(f"Requisições reais  → setlist.fm: {c['setlistfm']} | MusicBrainz: {c['musicbrainz']}")
    print(f"Lidos do cache     → setlist.fm: {h['setlistfm']} | MusicBrainz: {h['musicbrainz']}")

print("Configuração carregada com sucesso.")
print(f"MAX_PAGES = {MAX_PAGES}")

---
## Etapa 1 – MBIDs das Bandas no MusicBrainz

In [ ]:
reset_counters()

mbids = {}
rows = []

for banda in BANDAS:
    resultado = mbclient.search_artist(banda)
    artistas = resultado.get("artists", [])

    if not artistas:
        print(f"[AVISO] Nenhum resultado para '{banda}'")
        continue

    top = artistas[0]
    mbid = top.get("id", "")
    nome = top.get("name", "")
    pais = top.get("country") or "N/A"
    life = top.get("life-span", {})
    inicio = life.get("begin") or "N/A"
    fim = life.get("end") or "ativo"
    score = top.get("score", 0)

    mbids[banda] = mbid
    rows.append({
        "Banda (busca)": banda, "Nome MB": nome, "MBID": mbid,
        "País": pais, "Início": inicio, "Fim": fim, "Score": score,
    })

df_mbids = pd.DataFrame(rows)
print("\n=== MBIDs encontrados ===")
print(df_mbids.to_string(index=False))

_atualiza_totais()
_print_contadores("Etapa 1")

> **⚠ Confira manualmente** os MBIDs antes de continuar.

In [ ]:
# Correções manuais de MBID, se necessário.
# Exemplo: mbids["Oasis"] = "39ab1aed-75e0-4140-bd47-540276886b60"

print("MBIDs confirmados:")
for banda, mbid in mbids.items():
    print(f"  {banda:25s} → {mbid}")

---
## Etapa 2 – Coleta de Setlists

In [ ]:
reset_counters()

respostas_setlists: dict[str, list[dict]] = {}

for banda, mbid in mbids.items():
    respostas_setlists[banda] = []
    print(f"\n[{banda}]")

    for pagina in range(1, MAX_PAGES + 1):
        resp = slclient.get_artist_setlists(mbid=mbid, page=pagina)
        respostas_setlists[banda].append(resp)

        total_api = resp.get("total", "?")
        items_na_pagina = len(resp.get("setlist", []))
        print(f"  página {pagina}: {items_na_pagina} setlists (total na API: {total_api})")

        if items_na_pagina < resp.get("itemsPerPage", 20):
            print("  → Última página alcançada.")
            break

_atualiza_totais()
_print_contadores("Etapa 2")

---
## Etapa 3 – Normalização em DataFrames

Medleys marcados com ` / ` são separados: cada parte vira uma entrada própria com `is_medley = True`.

In [ ]:
rows_shows = []
rows_entradas = []

for banda, paginas in respostas_setlists.items():
    for resp in paginas:
        for show in resp.get("setlist", []):
            show_id = show.get("id", "")
            data_str = show.get("eventDate", "")  # DD-MM-YYYY
            tour = (show.get("tour") or {}).get("name", None)
            venue = show.get("venue", {})
            local_nome = venue.get("name", "")
            city_obj = venue.get("city", {})
            cidade = city_obj.get("name", "")
            pais = (city_obj.get("country") or {}).get("name", "")

            rows_shows.append({
                "show_id": show_id, "banda": banda,
                "data": pd.to_datetime(data_str, format="%d-%m-%Y", errors="coerce"),
                "turne": tour, "local": local_nome, "cidade": cidade, "pais": pais,
            })

            for set_idx, set_block in enumerate(show.get("sets", {}).get("set", [])):
                is_encore = bool(set_block.get("encore", 0))
                for pos, song in enumerate(set_block.get("song", []), start=1):
                    nome_raw = song.get("name", "")
                    is_cover = "cover" in song
                    is_tape  = song.get("tape", False)

                    # Separa medleys em partes (separador " / ")
                    partes = [p.strip() for p in nome_raw.split(" / ") if p.strip()]
                    is_medley = len(partes) > 1

                    for parte in (partes if is_medley else [nome_raw]):
                        rows_entradas.append({
                            "show_id": show_id, "banda": banda,
                            "musica": parte,
                            "posicao": pos, "set_idx": set_idx,
                            "is_bis": is_encore, "is_cover": is_cover,
                            "is_tape": is_tape, "is_medley": is_medley,
                        })

df_shows = pd.DataFrame(rows_shows)
df_entradas = pd.DataFrame(rows_entradas)

n_medley = df_entradas["is_medley"].sum()
print(f"df_shows: {len(df_shows)} linhas, {df_shows['banda'].nunique()} bandas")
print(f"df_entradas: {len(df_entradas)} linhas  (partes de medley: {n_medley})")
print("\nPrimeiras linhas de df_shows:")
display(df_shows.head())
print("\nPrimeiras linhas de df_entradas:")
display(df_entradas.head())

---
## Etapa 4 – Cobertura por Banda e por Ano

In [ ]:
shows_com_musica = set(df_entradas["show_id"].unique())

df_shows["tem_setlist"] = df_shows["show_id"].isin(shows_com_musica)
df_shows["ano"] = df_shows["data"].dt.year

cob_banda = (
    df_shows.groupby("banda")
    .agg(total_shows=("show_id", "count"), shows_com_setlist=("tem_setlist", "sum"))
    .assign(pct_preenchido=lambda d: (d["shows_com_setlist"] / d["total_shows"] * 100).round(1))
    .reset_index()
)

print("=== Cobertura por banda ===")
display(cob_banda)

cob_ano = (
    df_shows.dropna(subset=["ano"])
    .groupby(["banda", "ano"])
    .agg(total_shows=("show_id", "count"), shows_com_setlist=("tem_setlist", "sum"))
    .assign(pct_preenchido=lambda d: (d["shows_com_setlist"] / d["total_shows"] * 100).round(1))
    .reset_index()
)

bandas_unicas = cob_ano["banda"].unique()
fig, axes = plt.subplots(nrows=len(bandas_unicas), ncols=1,
                         figsize=(12, 3 * len(bandas_unicas)), sharex=False)

for ax, banda in zip(axes, bandas_unicas):
    sub = cob_ano[cob_ano["banda"] == banda].sort_values("ano")
    ax.bar(sub["ano"].astype(int), sub["pct_preenchido"], color="steelblue", alpha=0.8)
    ax.set_title(banda)
    ax.set_ylabel("% preenchido")
    ax.set_ylim(0, 105)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.axhline(50, color="red", linestyle="--", linewidth=0.8, alpha=0.5)

plt.suptitle("Cobertura de Setlists por Ano", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## Etapa 5 – Qualidade das Entradas

In [ ]:
total_entradas = len(df_entradas)

print("=== Qualidade global das entradas ===")
print(f"  Total de entradas : {total_entradas}")
print(f"  % covers          : {df_entradas['is_cover'].mean() * 100:.1f}%")
print(f"  % tape            : {df_entradas['is_tape'].mean() * 100:.1f}%")
print(f"  % medley          : {df_entradas['is_medley'].mean() * 100:.1f}%")
print(f"  % músicas sem nome: {(df_entradas['musica'].str.strip() == '').mean() * 100:.1f}%")

def _sem_turne(g):
    return pd.Series({"sem_turne": g["turne"].isna().sum(), "total": len(g)})

sem_turne = (
    df_shows.groupby("banda", group_keys=False)
    .apply(_sem_turne, include_groups=False)
    .reset_index()
)
sem_turne["pct_sem_turne"] = (sem_turne["sem_turne"] / sem_turne["total"] * 100).round(1)

qual_banda = (
    df_entradas.groupby("banda")
    .agg(
        total=("musica", "count"),
        covers=("is_cover", "sum"),
        tapes=("is_tape", "sum"),
        sem_nome=("musica", lambda s: (s.str.strip() == "").sum()),
    )
    .assign(
        pct_cover=lambda d: (d["covers"] / d["total"] * 100).round(1),
        pct_tape=lambda d: (d["tapes"] / d["total"] * 100).round(1),
        pct_sem_nome=lambda d: (d["sem_nome"] / d["total"] * 100).round(1),
    )
    .reset_index()
)

print("\n=== Qualidade por banda ===")
display(
    qual_banda[["banda", "pct_cover", "pct_tape", "pct_sem_nome"]]
    .merge(sem_turne[["banda", "sem_turne", "pct_sem_turne"]], on="banda")
)

---
## Etapa 6 – Álbuns de Estúdio no MusicBrainz

Lista de exclusão manual aplicada antes da etapa 6b.

In [ ]:
reset_counters()

# ---------------------------------------------------------------
# Lista de exclusão manual de álbuns (título exato por banda)
# ---------------------------------------------------------------
ALBUNS_EXCLUIR: dict[str, set[str]] = {
    "Oasis": {
        "Definitely Maybe Tour - 1994-12-18 - Manchester Academy - Homedown Showdown",
        "Eden Project 2009",
    },
    "Muse": {
        "The Resistance Instrumentals",
    },
    "Avenged Sevenfold": {
        "St. Louis, Mo, USA (17.02.09)",
    },
    "Metallica": {
        "Lulu",
    },
}

rows_albuns = []

for banda, mbid in mbids.items():
    print(f"[{banda}] buscando release-groups…")
    resp = mbclient.get_release_groups(mbid=mbid, offset=0)
    grupos = resp.get("release-groups", [])
    excluir = ALBUNS_EXCLUIR.get(banda, set())

    for rg in grupos:
        if rg.get("primary-type", "") != "Album" or rg.get("secondary-types", []):
            continue
        titulo = rg.get("title", "")
        if titulo in excluir:
            print(f"  [excluído] {titulo}")
            continue

        frd = rg.get("first-release-date") or None
        ano_alb = int(frd[:4]) if frd and len(frd) >= 4 and frd[:4].isdigit() else None

        rows_albuns.append({
            "banda": banda,
            "rg_mbid": rg.get("id", ""),
            "titulo": titulo,
            "first_release_date": frd,
            "ano_lancamento": ano_alb,
        })

df_albuns = pd.DataFrame(rows_albuns)

print(f"\nTotal de álbuns de estúdio após exclusão: {len(df_albuns)}")
print("\n=== Álbuns de estúdio por banda (revisão manual) ===")
df_albuns_sorted = df_albuns.sort_values(["banda", "ano_lancamento"])
print(df_albuns_sorted[["banda", "titulo", "ano_lancamento"]].to_string(index=False))

_atualiza_totais()
_print_contadores("Etapa 6")

---
## Etapa 6b – Faixas dos Álbuns de Estúdio

Para cada álbum, busca releases oficiais, escolhe o release representativo pelo
**número de faixas mais frequente (moda)** entre os releases daquele release-group,
preferindo a data mais antiga com essa moda; em empate de data, GB > US > XW > outros.

Resultado: `df_faixas_album`.

In [ ]:
# -------------------------------------------------------------------
# Função de normalização (reutilizada na Etapa 7)
# -------------------------------------------------------------------
def normalizar(texto: str) -> str:
    """
    1. Minúsculas
    2. Remove acentos (NFD)
    3. Remove conteúdo entre parênteses/colchetes
    4. Remove sufixos de versão após hífen
    5. Remove apóstrofos (' e ') sem espaço
    6. Substitui & por 'and'
    7. Remove pontuação restante
    8. Colapsa espaços
    """
    if not isinstance(texto, str):
        return ""
    t = texto.lower()
    t = unicodedata.normalize("NFD", t)
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")
    t = re.sub(r"[\(\[][^)\]]*[\)\]]", "", t)
    for sfx in [
        r"\s*-\s*remaster(ed)?.*$", r"\s*-\s*live.*$", r"\s*-\s*ao vivo.*$",
        r"\s*-\s*feat\.?.*$", r"\s*-\s*ft\.?.*$",
        r"\s*-\s*(\d{4}\s+)?version.*$", r"\s*-\s*single.*$", r"\s*-\s*bonus.*$",
    ]:
        t = re.sub(sfx, "", t)
    t = t.replace("\u2019", "").replace("\u2018", "").replace("'", "")
    t = re.sub(r"\s*&\s*", " and ", t)
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


# -------------------------------------------------------------------
# Escolha do release representativo por moda de faixas
# -------------------------------------------------------------------
_PAIS_PRIO = {"GB": 0, "US": 1, "XW": 2}

def _escolher_release(releases: list[dict]) -> tuple[dict | None, int]:
    """
    Retorna (release_escolhido, n_faixas_moda).
    Estratégia:
    1. Conta o total de faixas de cada release (soma de todas as mídias).
    2. Calcula a moda desse número entre todos os releases.
    3. Filtra candidatos com esse número de faixas.
    4. Entre eles, escolhe o de menor data; em empate, GB > US > XW > outros.
    """
    candidatos = []
    for r in releases:
        n = sum(len(m.get("tracks", [])) for m in r.get("media", []))
        if n > 0:
            candidatos.append((r, n))

    if not candidatos:
        return None, 0

    # Moda do número de faixas
    from collections import Counter
    contagem = Counter(n for _, n in candidatos)
    moda = contagem.most_common(1)[0][0]

    finalistas = [(r, n) for r, n in candidatos if n == moda]

    def _chave(rn):
        r, _ = rn
        data = r.get("date") or "9999"
        partes = data.split("-")
        data_norm = "-".join([
            partes[0].zfill(4) if len(partes) > 0 else "9999",
            partes[1].zfill(2) if len(partes) > 1 else "99",
            partes[2].zfill(2) if len(partes) > 2 else "99",
        ])
        pais_ord = _PAIS_PRIO.get((r.get("country") or "").upper(), 99)
        return (data_norm, pais_ord)

    melhor, _ = min(finalistas, key=_chave)
    return melhor, moda


reset_counters()
rows_faixas = []

for _, row in df_albuns.iterrows():
    banda      = row["banda"]
    rg_mbid    = row["rg_mbid"]
    album_tit  = row["titulo"]
    ano_alb    = row["ano_lancamento"]

    resp = mbclient.get_releases_for_release_group(rg_mbid)
    releases = resp.get("releases", [])

    release_rep, n_faixas_moda = _escolher_release(releases)
    if release_rep is None:
        print(f"  [AVISO] Sem releases com faixas: '{album_tit}' ({banda})")
        continue

    print(f"  {banda} / {album_tit} ({ano_alb}) — {n_faixas_moda} faixas (moda)")

    for medium in release_rep.get("media", []):
        for track in medium.get("tracks", []):
            titulo_faixa = (track.get("title") or "").strip()
            if not titulo_faixa:
                continue
            rows_faixas.append({
                "banda": banda,
                "album": album_tit,
                "ano_lancamento": ano_alb,
                "titulo_faixa": titulo_faixa,
                "titulo_normalizado": normalizar(titulo_faixa),
            })

df_faixas_album = pd.DataFrame(rows_faixas)

print(f"\nTotal de faixas de álbuns: {len(df_faixas_album)}")
print(f"Faixas únicas normalizadas: {df_faixas_album['titulo_normalizado'].nunique()}")

_atualiza_totais()
_print_contadores("Etapa 6b")

---
## Etapa 6c – Gravações no MusicBrainz

In [ ]:
reset_counters()

_RE_META = re.compile(r'^[\(\[\s][^\)\]]*[\)\]\s]*$')

rows_gravacoes = []

for banda, mbid in mbids.items():
    print(f"[{banda}] buscando gravações…")
    offset = 0
    guard  = 0

    while guard < 500:
        resp = mbclient.get_recordings(mbid=mbid, offset=offset)
        gravacoes = resp.get("recordings", [])
        total = resp.get("recording-count", 0)

        if not gravacoes:
            break

        for rec in gravacoes:
            titulo = rec.get("title", "").strip()
            if not titulo or _RE_META.match(titulo):
                continue
            frd = rec.get("first-release-date") or None
            ano_rec = int(frd[:4]) if frd and len(frd) >= 4 and frd[:4].isdigit() else None
            rows_gravacoes.append({
                "banda": banda, "rec_mbid": rec.get("id", ""),
                "titulo": titulo, "first_release_date": frd, "ano_lancamento": ano_rec,
            })

        offset += len(gravacoes)
        guard  += 1
        print(f"  {offset}/{total} gravações coletadas")
        if offset >= total:
            break

df_gravacoes_raw = pd.DataFrame(rows_gravacoes)
print(f"\nTotal bruto de gravações: {len(df_gravacoes_raw)}")

_atualiza_totais()
_print_contadores("Etapa 6c")

---
## Etapa 7 – Casamento de Nomes

- Normalização aplicada nos dois lados.
- Medleys: cada parte é comparada individualmente.
- Casamento com **gravações** e com **faixas de álbuns de estúdio**.
- Lista músicas em gravações mas fora de qualquer álbum (singles, EPs, lados B, covers não marcados).

In [ ]:
# Normaliza entradas de setlist (partes de medley já separadas na etapa 3)
df_entradas["musica_norm"] = df_entradas["musica"].apply(normalizar)

# Normaliza e deduplica gravações
df_gravacoes_raw["titulo_norm"] = df_gravacoes_raw["titulo"].apply(normalizar)
df_gravacoes = (
    df_gravacoes_raw[df_gravacoes_raw["titulo_norm"] != ""]
    .sort_values("ano_lancamento")
    .drop_duplicates(subset=["banda", "titulo_norm"], keep="first")
    .reset_index(drop=True)
)
print(f"Gravações após deduplicação: {len(df_gravacoes)}")

# Lookups por banda
gravacoes_por_banda: dict[str, set] = (
    df_gravacoes.groupby("banda")["titulo_norm"].apply(set).to_dict()
)

# Faixas de álbuns: mantém o álbum mais antigo por título normalizado
faixas_por_banda: dict[str, dict[str, str]] = {}
for banda_f in BANDAS:
    sub = (
        df_faixas_album[df_faixas_album["banda"] == banda_f]
        .sort_values("ano_lancamento")
        .drop_duplicates(subset=["titulo_normalizado"], keep="first")
    )
    faixas_por_banda[banda_f] = dict(zip(sub["titulo_normalizado"], sub["album"]))

rows_casamento   = []
rows_sem_rec     = []   # não casaram com gravações
rows_rec_sem_alb = []   # gravações mas fora de álbuns

for banda in BANDAS:
    musicas_sl = (
        df_entradas[
            (df_entradas["banda"] == banda)
            & (df_entradas["musica_norm"] != "")
            & (~df_entradas["is_cover"])
        ]["musica_norm"]
        .unique()
    )

    grav_mb  = gravacoes_por_banda.get(banda, set())
    faixas   = faixas_por_banda.get(banda, {})
    total_sl = len(musicas_sl)

    casaram_rec = sum(1 for m in musicas_sl if m in grav_mb)
    casaram_alb = sum(1 for m in musicas_sl if m in faixas)
    nao_rec     = [m for m in musicas_sl if m not in grav_mb]
    rec_sem_alb = [m for m in musicas_sl if m in grav_mb and m not in faixas]

    pct_rec = round(casaram_rec / total_sl * 100, 1) if total_sl > 0 else 0.0
    pct_alb = round(casaram_alb / total_sl * 100, 1) if total_sl > 0 else 0.0

    rows_casamento.append({
        "banda": banda, "musicas_distintas_sl": total_sl,
        "casaram_gravacoes": casaram_rec, "pct_gravacoes": pct_rec,
        "casaram_albuns": casaram_alb, "pct_albuns": pct_alb,
    })
    for m in sorted(nao_rec)[:20]:
        rows_sem_rec.append({"banda": banda, "musica_norm": m})
    for m in sorted(rec_sem_alb):
        rows_rec_sem_alb.append({"banda": banda, "musica_norm": m})

df_casamento    = pd.DataFrame(rows_casamento)
df_sem_rec      = pd.DataFrame(rows_sem_rec)
df_rec_sem_alb  = pd.DataFrame(rows_rec_sem_alb)

print("\n=== Casamento por banda ===")
display(df_casamento)
print(df_casamento.to_string(index=False))

print("\n=== Até 20 músicas que NÃO casaram com gravações, por banda ===")
for banda in BANDAS:
    sub = df_sem_rec[df_sem_rec["banda"] == banda]["musica_norm"].tolist()
    if sub:
        print(f"\n[{banda}]")
        for m in sub:
            print(f"  - {m}")

print("\n=== Gravações sem álbum de estúdio (singles/EPs/lados B), por banda ===")
for banda in BANDAS:
    sub = df_rec_sem_alb[df_rec_sem_alb["banda"] == banda]["musica_norm"].tolist()
    if sub:
        print(f"\n[{banda}] ({len(sub)} músicas)")
        for m in sub:
            print(f"  - {m}")

---
## Etapa 8 – Intervalos Entre Shows e Análise de Abandono

**8a – Distribuição de intervalos** entre shows consecutivos (histograma + limiares de 6/12/18 meses).

**8b – Músicas 'abandonadas'** com a regra: ausente nos próximos N shows da banda, para N = 25, 50 e 100.
Músicas cuja última aparição está a menos de N shows do fim do histórico entram como *censuradas* (não abandonadas).

In [ ]:
# ---- 8a: Intervalos ----
LIMIARES = [(6, 182), (12, 365), (18, 548)]

rows_intervalos = []
fig, axes = plt.subplots(nrows=len(BANDAS), ncols=1,
                         figsize=(12, 3 * len(BANDAS)), sharex=False)

for ax, banda in zip(axes, BANDAS):
    shows_banda = (
        df_shows[df_shows["banda"] == banda]
        .dropna(subset=["data"])
        .sort_values("data")
    )
    if len(shows_banda) < 2:
        ax.set_title(f"{banda} (dados insuficientes)")
        continue

    intervalos = shows_banda["data"].diff().dt.days.dropna()
    intervalos = intervalos[intervalos > 0]

    for meses, dias in LIMIARES:
        rows_intervalos.append({
            "banda": banda, "limiar_meses": meses,
            "pausas_acima": int((intervalos > dias).sum()),
        })

    ax.hist(intervalos, bins=30, color="steelblue", alpha=0.8, edgecolor="white")
    for (meses, dias), cor in zip(LIMIARES, ["orange", "red", "darkred"]):
        ax.axvline(dias, color=cor, linestyle="--", linewidth=1, label=f"{meses} meses")
    ax.set_title(f"{banda} — intervalos entre shows")
    ax.set_xlabel("dias")
    ax.set_ylabel("frequência")
    ax.legend(fontsize=8)

plt.suptitle("Distribuição de Intervalos Entre Shows Consecutivos", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

df_intervalos = pd.DataFrame(rows_intervalos)
df_pivot_pausas = (
    df_intervalos
    .pivot(index="banda", columns="limiar_meses", values="pausas_acima")
    .rename(columns={m: f"pausas > {m}m" for m, _ in LIMIARES})
)
print("\n=== Pausas prolongadas por banda ===")
display(df_pivot_pausas)

In [ ]:
# ---- 8b: Análise de abandono por janela de N shows ----
# Para cada banda:
#   - Ordena os shows da banda por data (histórico cronológico).
#   - Numera os shows de 1 a T (T = total de shows com setlist).
#   - Para cada música, encontra o índice do último show em que apareceu.
#   - "Abandonada" com janela N: last_idx <= T - N
#   - "Censurada" (ainda pode aparecer): last_idx > T - N

NS_JANELA = [25, 50, 100]

rows_abandono = []

for banda in BANDAS:
    # Shows com setlist, ordenados por data (mais antigo primeiro)
    shows_ord = (
        df_shows[
            (df_shows["banda"] == banda) &
            (df_shows["show_id"].isin(shows_com_musica)) &
            (df_shows["data"].notna())
        ]
        .sort_values("data")
        .reset_index(drop=True)
    )
    T = len(shows_ord)  # total de shows com setlist
    if T == 0:
        continue

    # Mapa show_id → índice cronológico (1-based)
    show_idx = {row["show_id"]: i + 1 for i, row in shows_ord.iterrows()}

    # Músicas da banda (excluindo covers, tapes e vazias)
    ent_banda = df_entradas[
        (df_entradas["banda"] == banda) &
        (~df_entradas["is_cover"]) &
        (~df_entradas["is_tape"]) &
        (df_entradas["musica_norm"] != "")
    ].copy()

    ent_banda["show_idx"] = ent_banda["show_id"].map(show_idx)
    ent_banda = ent_banda.dropna(subset=["show_idx"])

    # Último show de cada música
    ultimo_idx = (
        ent_banda.groupby("musica_norm")["show_idx"]
        .max()
        .reset_index()
        .rename(columns={"show_idx": "last_idx"})
    )

    row = {"banda": banda, "total_shows_com_setlist": T, "musicas_distintas": len(ultimo_idx)}
    for N in NS_JANELA:
        if T < N:
            row[f"abandonadas_N{N}"] = None  # histórico curto demais
            row[f"censuradas_N{N}"] = None
        else:
            corte = T - N
            abandonadas = (ultimo_idx["last_idx"] <= corte).sum()
            censuradas  = (ultimo_idx["last_idx"] >  corte).sum()
            row[f"abandonadas_N{N}"] = int(abandonadas)
            row[f"censuradas_N{N}"]  = int(censuradas)
    rows_abandono.append(row)

df_abandono = pd.DataFrame(rows_abandono)

print("=== Músicas abandonadas por janela de N shows ===")
print("(abandonada = não apareceu nos últimos N shows da banda; censurada = ainda pode aparecer)\n")
display(df_abandono)
print(df_abandono.to_string(index=False))

---
## Etapa 9 – Tabela Resumo Final

In [ ]:
def avaliar(pct_setlist: float, pct_casamento: float) -> str:
    if pct_setlist >= 70 and pct_casamento >= 70:
        return "boa"
    elif pct_setlist >= 40 or pct_casamento >= 40:
        return "média"
    else:
        return "fraca"


maior_pausa_por_banda: dict[str, float] = {}
for banda in BANDAS:
    shows_b = (
        df_shows[df_shows["banda"] == banda]
        .dropna(subset=["data"]).sort_values("data")
    )
    if len(shows_b) < 2:
        maior_pausa_por_banda[banda] = 0.0
        continue
    ivs = shows_b["data"].diff().dt.days.dropna()
    ivs = ivs[ivs > 0]
    maior_pausa_por_banda[banda] = round(ivs.max() / 30, 1) if len(ivs) > 0 else 0.0

cob_idx = cob_banda.set_index("banda")
cas_idx = df_casamento.set_index("banda")

rows_resumo = []
for banda in BANDAS:
    total_shows  = int(cob_idx.loc[banda, "total_shows"]) if banda in cob_idx.index else 0
    pct_setlist  = float(cob_idx.loc[banda, "pct_preenchido"]) if banda in cob_idx.index else 0.0
    pct_cas      = float(cas_idx.loc[banda, "pct_gravacoes"]) if banda in cas_idx.index else 0.0
    maior_pausa  = maior_pausa_por_banda.get(banda, 0.0)

    rows_resumo.append({
        "Banda": banda,
        "Total Shows": total_shows,
        "% c/ Setlist": pct_setlist,
        "% Casamento MB": pct_cas,
        "Maior Pausa (meses)": maior_pausa,
        "Avaliação": avaliar(pct_setlist, pct_cas),
    })

df_resumo = pd.DataFrame(rows_resumo)

print("=" * 70)
print("TABELA RESUMO FINAL – Encore Fase 0")
print("=" * 70)

display(
    df_resumo.style
    .format({"% c/ Setlist": "{:.1f}%", "% Casamento MB": "{:.1f}%"})
    .map(
        lambda v: "background-color: #d4edda" if v == "boa"
        else ("background-color: #fff3cd" if v == "média" else "background-color: #f8d7da"),
        subset=["Avaliação"],
    )
)

print(df_resumo.to_string(index=False))

# Totais acumulados da sessão (não zeram entre células)
print(f"\nTotal acumulado de requisições reais na sessão:")
print(f"  setlist.fm:   {_total_req['setlistfm']}")
print(f"  MusicBrainz:  {_total_req['musicbrainz']}")
print(f"\nTotal acumulado de leituras de cache na sessão:")
print(f"  setlist.fm:   {_total_cache['setlistfm']}")
print(f"  MusicBrainz:  {_total_cache['musicbrainz']}")